# Notebook Overview — Evaluate Development Results

## Purpose

This notebook evaluates development-subset VideoQA results generated by previous project notebooks and produces quantitative performance metrics, analysis summaries, visualizations, and reporting artifacts.

Because NExT-QA is a multiple-choice VideoQA benchmark, multiple-choice accuracy is the primary evaluation metric when experiment outputs use `answer_mode = "multiple_choice"`. Exact-match and partial-match text comparisons are retained as secondary diagnostic metrics that provide additional insight into model-generated responses but are not considered the primary benchmark score.

During the current development phase, this notebook evaluates baseline VideoQA results generated by Notebook 01. The notebook is designed to support future comparative evaluation of baseline, pretrained-representation, and autoencoder-based VideoQA experiments using a common evaluation framework.

Evaluation procedures include prediction validation, multiple-choice accuracy analysis, reasoning-category analysis, question-type analysis, runtime analysis, visualization generation, and experiment reporting.

## Inputs

* VideoQA prediction results
* Experiment summary reports
* Runtime statistics
* NExT-QA validation annotations
* Project configuration settings

## Outputs

* Multiple-choice evaluation metrics
* Prediction verification summaries
* Choice prediction accuracy summaries
* Reasoning-category performance analyses
* Question-type performance analyses
* Answer-length analyses
* Runtime analyses
* Performance visualizations
* Evaluation reports
* Saved reporting artifacts

## Workflow

The workflow begins by loading experiment outputs and reference annotation data. Input files and required prediction columns are validated before evaluation datasets are prepared.

Prediction results are matched with NExT-QA annotation records to support reasoning-category and question-type analysis. For multiple-choice experiments, predicted answer choices are compared against ground-truth answer choices to compute benchmark accuracy metrics. Exact-match and partial-match text metrics are also computed as secondary diagnostic measures.

Evaluation metrics, answer-length statistics, runtime summaries, category-level analyses, and visualization artifacts are generated and saved for later reporting and experiment comparison.

The notebook provides a common evaluation framework that can be applied consistently across baseline, pretrained-representation, and autoencoder-based VideoQA experiments.

## Notes

This notebook does not perform VideoQA inference and does not require access to raw video files. Evaluation is performed using saved experiment outputs and NExT-QA reference annotations.

For NExT-QA multiple-choice experiments, choice accuracy is the primary benchmark metric. Exact-match and partial-match text comparisons are retained for diagnostic analysis only. Future work may incorporate semantic similarity metrics and additional benchmark measures to complement the current evaluation framework.


### 🔷 Step 1 — Initialize Evaluation Environment

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files, configuration modules, and dataset resources are available for evaluation.
* Prepare the notebook environment for loading saved experiment results and evaluation reference data.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Initialize Environment and Restore Dataset
# ============================================================

VERBOSE = True
REQUIRE_L4_GPU = False
EXPECTED_NEXTQA_VIDEO_COUNT = 5440

import os
import shutil
import time
from pathlib import Path

import pandas as pd

from google.colab import userdata, drive

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# Clone Required Repository Files
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# Load Project Configuration and Utility Modules
# ------------------------------------------------------------

print("\nLoading project configuration and utility modules...")

from src.videoqa_representation_config import *

from src.nextqa_video_cache import *
from src.nextqa_metadata import *
from src.video_segments import *
from src.training_validation import *
from src.training_metadata_io import *

required_paths = [
    Path("src"),
    Path("datasets"),
    Path("outputs"),
    QUESTIONS_DIR,
    METADATA_DIR,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    for path in missing_paths:
        print(f"Missing required path: {path}")

    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

for output_dir in [
    TRAINING_METADATA_DIR,
    TRAINING_REPORTS_DIR,
]:
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# Restore Local NExT-QA Video Cache
# ------------------------------------------------------------

print("\nChecking local NExT-QA video cache...")

existing_video_files = sorted(
    VIDEOS_DIR.rglob("*.mp4")
)

if len(existing_video_files) == EXPECTED_NEXTQA_VIDEO_COUNT:

    video_cache_restore_summary = {
        "cache_status": "already_available",
        "video_count": len(existing_video_files),
        "local_videos_dir": str(VIDEOS_DIR),
        "archive_mode": "not_required",
        "verified": True,
    }

    print("Local video cache already available.")
    print(f"Videos found: {len(existing_video_files):,}")

else:

    print("Local video cache missing or incomplete.")
    print(f"Videos found locally: {len(existing_video_files):,}")
    print("Restoring videos from Google Drive...")

    GOOGLE_DRIVE_MOUNT = "/content/drive"

    if not os.path.exists(GOOGLE_DRIVE_MOUNT):
        print("Mounting Google Drive...")
        drive.mount(GOOGLE_DRIVE_MOUNT)
    else:
        print("Google Drive already mounted.")

    drive_root = Path(GOOGLE_DRIVE_MOUNT) / "MyDrive"

    if not drive_root.exists():
        raise FileNotFoundError(
            "Unable to access Google Drive root directory."
        )

    DRIVE_DATASET_DIR = drive_root / "VideoQA_Project" / "NExT-QA"
    DRIVE_RELEASES_DIR = DRIVE_DATASET_DIR / "releases"

    LOCAL_ARCHIVE_DIR = DATASET_DIR / "archives"
    LOCAL_ARCHIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    COMBINED_ARCHIVE_NAME = "NExTVideo_combined.zip"

    DRIVE_COMBINED_ARCHIVE_PATH = (
        DRIVE_RELEASES_DIR /
        COMBINED_ARCHIVE_NAME
    )

    COMBINED_ARCHIVE_PATH = (
        LOCAL_ARCHIVE_DIR /
        COMBINED_ARCHIVE_NAME
    )

    required_archive_files = [
        "NExTVideo.z01",
        "NExTVideo.z02",
        "NExTVideo.z03",
        "NExTVideo.z04",
        "NExTVideo.z05",
        "NExTVideo.z06",
        "NExTVideo.zip",
    ]

    if not DRIVE_RELEASES_DIR.exists():
        raise FileNotFoundError(
            "Google Drive NExT-QA releases directory not found:\n"
            f"{DRIVE_RELEASES_DIR}"
        )

    if DRIVE_COMBINED_ARCHIVE_PATH.exists():

        print("Preferred combined archive found.")

        source_size = DRIVE_COMBINED_ARCHIVE_PATH.stat().st_size
        copy_start_time = time.time()

        if COMBINED_ARCHIVE_PATH.exists():
            local_size = COMBINED_ARCHIVE_PATH.stat().st_size

            if local_size == source_size:
                print("Local archive already exists. Copy skipped.")
            else:
                print("Replacing incomplete local archive.")
                COMBINED_ARCHIVE_PATH.unlink()
                shutil.copy2(
                    DRIVE_COMBINED_ARCHIVE_PATH,
                    COMBINED_ARCHIVE_PATH,
                )

        else:
            print("Copying archive to local runtime...")
            shutil.copy2(
                DRIVE_COMBINED_ARCHIVE_PATH,
                COMBINED_ARCHIVE_PATH,
            )

        copy_elapsed_time = time.time() - copy_start_time
        local_size = COMBINED_ARCHIVE_PATH.stat().st_size

        if local_size != source_size:
            raise ValueError(
                "Combined archive copy failed size verification."
            )

        archive_restore_summary = {
            "archive_mode": "combined",
            "source_archive": str(DRIVE_COMBINED_ARCHIVE_PATH),
            "local_archive": str(COMBINED_ARCHIVE_PATH),
            "archive_size_gb": local_size / (1024 ** 3),
            "copy_elapsed_seconds": copy_elapsed_time,
            "verified": True,
        }

        print(f"Archive ready: {local_size / (1024 ** 3):.2f} GB")
        print(f"Copy time: {copy_elapsed_time:.1f} seconds")

    else:

        print("Combined archive not found.")
        print("Using legacy multipart archive workflow...")

        archive_verification_summary = verify_nextqa_archive_parts(
            archive_parts_dir=DRIVE_RELEASES_DIR,
            required_archive_files=required_archive_files,
            verbose=VERBOSE,
        )

        local_archive_summary = copy_nextqa_archive_parts_to_local(
            source_archive_dir=DRIVE_RELEASES_DIR,
            local_archive_dir=LOCAL_ARCHIVE_DIR,
            required_archive_files=required_archive_files,
            verbose=VERBOSE,
        )

        archive_restore_summary = build_combined_nextqa_archive(
            local_archive_dir=LOCAL_ARCHIVE_DIR,
            combined_archive_path=COMBINED_ARCHIVE_PATH,
            split_archive_name="NExTVideo.zip",
            required_archive_files=required_archive_files,
            force_rebuild=True,
            verbose=VERBOSE,
        )

    print("Extracting or verifying local video cache...")

    extraction_start_time = time.time()

    extract_summary = extract_nextqa_video_archive(
        combined_archive_path=COMBINED_ARCHIVE_PATH,
        local_videos_dir=VIDEOS_DIR,
        force_extract=False,
        verbose=VERBOSE,
    )

    extraction_elapsed_time = time.time() - extraction_start_time

    restored_video_files = sorted(
        VIDEOS_DIR.rglob("*.mp4")
    )

    if len(restored_video_files) != EXPECTED_NEXTQA_VIDEO_COUNT:
        raise ValueError(
            "NExT-QA video cache verification failed. "
            f"Expected {EXPECTED_NEXTQA_VIDEO_COUNT:,} videos, "
            f"found {len(restored_video_files):,}."
        )

    video_cache_restore_summary = {
        "cache_status": "restored",
        "video_count": len(restored_video_files),
        "local_videos_dir": str(VIDEOS_DIR),
        "archive_mode": archive_restore_summary.get(
            "archive_mode",
            "unknown",
        ),
        "archive_restore_summary": archive_restore_summary,
        "extract_summary": extract_summary,
        "extraction_elapsed_seconds": extraction_elapsed_time,
        "verified": True,
    }

    print("Video cache restored.")
    print(f"Videos found: {len(restored_video_files):,}")
    print(f"Extraction time: {extraction_elapsed_time:.1f} seconds")

print("Local NExT-QA video cache ready.")

# ------------------------------------------------------------
# Load NExT-QA Metadata and Video Inventory
# ------------------------------------------------------------

print("\nLoading NExT-QA metadata and video inventory...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

video_inventory_df = build_nextqa_video_inventory(
    videos_dir=VIDEOS_DIR,
    verbose=VERBOSE,
)

annotations_with_videos_df = (
    attach_video_inventory_to_annotations(
        annotations=annotations_df,
        video_inventory=video_inventory_df,
        verbose=VERBOSE,
    )
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

coverage_summary = verify_annotation_video_coverage(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

print("\nDataset metadata ready.")
print(f"Annotation records: {len(annotations_df):,}")
print(f"Video inventory   : {len(video_inventory_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment initialization complete.")
print("-" * 60)
print("Notebook is ready for baseline VideoQA inference.")



Initializing notebook environment...
------------------------------------------------------------
Cloning project repository...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 26 (delta 0), reused 0 (delta 0), pack-reused 23 (from 2)
Receiving objects: 100% (26/26), 2.71 MiB | 22.97 MiB/s, done.
Repository ready: /content/videoqa-representation-comparison

Loading project configuration and utility modules...
Configuration loaded.
Project paths initialized.

Checking local NExT-QA video cache...
Local video cache missing or incomplete.
Videos found locally: 0
Restoring videos from Google Drive...
Mounting Google Drive...
Mounted at /content/drive
Preferred combined archive found.
Copying archive to local runtime...


### 🔷 Step 2 — Load Evaluation Data

* Load VideoQA prediction results generated by previous experiment notebooks.
* Load experiment summary statistics and runtime metrics.
* Load NExT-QA validation annotations required for evaluation, category analysis, and reporting.
* Verify that all required evaluation input files are available and accessible.
* Display dataset sizes, column information, and file statistics to confirm successful loading.
* Preview loaded prediction, summary, and annotation data prior to evaluation.

In [ ]:
# ============================================================
# Step 2: Load Evaluation Data
# ============================================================

import pandas as pd

from src.videoqa_representation_config import *

print("Loading evaluation data...\n")

# ------------------------------------------------------------
# Input Files
# ------------------------------------------------------------

VALIDATION_ANNOTATIONS_CSV = (
    QUESTIONS_DIR / "val.csv"
)

required_input_files = [
    BASELINE_PREDICTIONS_CSV,
    BASELINE_SUMMARY_CSV,
    VALIDATION_ANNOTATIONS_CSV,
]

missing_input_files = [
    file_path
    for file_path in required_input_files
    if not file_path.exists()
]

if missing_input_files:
    for file_path in missing_input_files:
        print(f"Missing required input file: {file_path}")

    raise FileNotFoundError(
        "One or more required evaluation input files are missing."
    )

# ------------------------------------------------------------
# Load Baseline Experiment Results
# ------------------------------------------------------------

baseline_predictions_df = pd.read_csv(
    BASELINE_PREDICTIONS_CSV
)

baseline_summary_df = pd.read_csv(
    BASELINE_SUMMARY_CSV
)

# ------------------------------------------------------------
# Load Evaluation Reference Data
# ------------------------------------------------------------

val_annotations_df = pd.read_csv(
    VALIDATION_ANNOTATIONS_CSV
)

# ------------------------------------------------------------
# Display Dataset Information
# ------------------------------------------------------------

print("Loaded Evaluation Data")
print("-" * 60)
print(
    f"Baseline Predictions      : "
    f"{len(baseline_predictions_df):,} records"
)
print(
    f"Baseline Summary          : "
    f"{len(baseline_summary_df):,} records"
)
print(
    f"Validation Annotations    : "
    f"{len(val_annotations_df):,} records"
)

print("\nInput Files")
print("-" * 60)
for file_path in required_input_files:
    print(file_path)

# ------------------------------------------------------------
# Display Available Columns
# ------------------------------------------------------------

print("\nBaseline Prediction Columns")
print("-" * 60)
print(list(baseline_predictions_df.columns))

print("\nBaseline Summary Columns")
print("-" * 60)
print(list(baseline_summary_df.columns))

print("\nValidation Annotation Columns")
print("-" * 60)
print(list(val_annotations_df.columns))

# ------------------------------------------------------------
# Preview Loaded Data
# ------------------------------------------------------------

print("\nBaseline Predictions Preview")
display(baseline_predictions_df.head())

print("\nBaseline Summary Preview")
display(baseline_summary_df.head())



### 🔷 Step 3 — Validate Evaluation Inputs

* Verify that all required evaluation datasets were loaded successfully.
* Confirm required columns exist in prediction, summary, and annotation datasets.
* Validate dataset record counts and structural integrity.
* Verify the presence of required evaluation metrics and reporting fields.
* Identify missing columns, missing metrics, and other input validation issues.
* Report validation results before evaluation processing begins.


In [ ]:
# ============================================================
# Step 3: Validate Evaluation Inputs
# ============================================================

print("Validating evaluation inputs...\n")

# ------------------------------------------------------------
# Verify Required DataFrames Exist
# ------------------------------------------------------------

required_dataframes = [
    "baseline_predictions_df",
    "baseline_summary_df",
    "val_annotations_df",
]

missing_dataframes = [
    dataframe_name
    for dataframe_name in required_dataframes
    if dataframe_name not in globals()
]

if missing_dataframes:
    raise NameError(
        "Required DataFrames are missing. Run Step 2 first:\n"
        + "\n".join(missing_dataframes)
    )

# ------------------------------------------------------------
# Validate Baseline Prediction Columns
# ------------------------------------------------------------

required_prediction_columns = [
    "video",
    "question",
    "ground_truth",
    "prediction",
    "answer_mode",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in baseline_predictions_df.columns
]

if missing_prediction_columns:
    raise ValueError(
        "Baseline prediction results are missing required columns:\n"
        + "\n".join(missing_prediction_columns)
    )

# ------------------------------------------------------------
# Validate Baseline Summary Columns
# ------------------------------------------------------------

required_summary_columns = [
    "metric",
    "value",
]

missing_summary_columns = [
    column
    for column in required_summary_columns
    if column not in baseline_summary_df.columns
]

if missing_summary_columns:
    raise ValueError(
        "Baseline summary is missing required columns:\n"
        + "\n".join(missing_summary_columns)
    )

# ------------------------------------------------------------
# Validate Multiple-Choice Summary Metrics
# ------------------------------------------------------------

required_multiple_choice_summary_metrics = [
    "answer_mode",
    "valid_choice_predictions",
    "invalid_choice_predictions",
    "correct_choice_predictions",
    "choice_accuracy",
]

summary_metrics = set(
    baseline_summary_df["metric"]
    .astype(str)
    .tolist()
)

missing_summary_metrics = [
    metric
    for metric in required_multiple_choice_summary_metrics
    if metric not in summary_metrics
]

if missing_summary_metrics:
    raise ValueError(
        "Baseline summary is missing required multiple-choice metrics:\n"
        + "\n".join(missing_summary_metrics)
    )

# ------------------------------------------------------------
# Validate Validation Annotation Columns
# ------------------------------------------------------------

required_annotation_columns = [
    "video",
    "question",
    "answer",
    "type",
]

missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in val_annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        "Validation annotations are missing required columns:\n"
        + "\n".join(missing_annotation_columns)
    )

# ------------------------------------------------------------
# Validate Annotation Answer-Choice Columns
# ------------------------------------------------------------

required_annotation_choice_columns = [
    "a0",
    "a1",
    "a2",
    "a3",
    "a4",
]

missing_annotation_choice_columns = [
    column
    for column in required_annotation_choice_columns
    if column not in val_annotations_df.columns
]

if missing_annotation_choice_columns:
    raise ValueError(
        "Validation annotations are missing required answer-choice columns:\n"
        + "\n".join(missing_annotation_choice_columns)
    )

# ------------------------------------------------------------
# Validate Answer Mode
# ------------------------------------------------------------

answer_modes = sorted(
    baseline_predictions_df["answer_mode"]
    .dropna()
    .astype(str)
    .unique()
)

if not answer_modes:
    raise ValueError(
        "No answer_mode values found in baseline predictions."
    )

# ------------------------------------------------------------
# Validate Choice Values
# ------------------------------------------------------------

valid_choice_values = {
    "0",
    "1",
    "2",
    "3",
    "4",
}

ground_truth_choice_values = set(
    baseline_predictions_df["ground_truth_choice"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

predicted_choice_values = set(
    baseline_predictions_df["predicted_choice"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

invalid_ground_truth_choices = sorted(
    ground_truth_choice_values - valid_choice_values
)

invalid_predicted_choices = sorted(
    predicted_choice_values - valid_choice_values
)

if invalid_ground_truth_choices:
    raise ValueError(
        "Invalid ground_truth_choice values found:\n"
        + "\n".join(invalid_ground_truth_choices)
    )

# Invalid predicted choices are not fatal because they reflect model behavior.
# They are counted and reported as part of evaluation.

# ------------------------------------------------------------
# Report Validation Summary
# ------------------------------------------------------------

print("Evaluation input validation complete.")
print("-" * 60)
print(f"Prediction records        : {len(baseline_predictions_df):,}")
print(f"Summary records           : {len(baseline_summary_df):,}")
print(f"Validation annotations    : {len(val_annotations_df):,}")
print(f"Answer modes found        : {answer_modes}")

if "multiple_choice" in answer_modes:
    print("Primary evaluation metric : multiple-choice accuracy")
else:
    print("Primary evaluation metric : exact match")

print("\nChoice Validation")
print("-" * 60)
print(
    f"Invalid ground-truth choices : "
    f"{len(invalid_ground_truth_choices)}"
)
print(
    f"Invalid predicted choices    : "
    f"{len(invalid_predicted_choices)}"
)

if invalid_predicted_choices:
    print("\nInvalid predicted choice values found:")
    for value in invalid_predicted_choices:
        print(f"  {value}")



### 🔷 Step 4 — Prepare Evaluation Dataset

* Create a unified evaluation dataset from VideoQA prediction results and NExT-QA annotation records.
* Standardize prediction, question, answer, and metadata fields for evaluation processing.
* Attach question-type information and reasoning-category assignments.
* Generate unique evaluation identifiers for each prediction record.
* Match prediction results with corresponding NExT-QA annotation entries.
* Summarize evaluation records, category distributions, and question-type distributions.
* Prepare the evaluation dataset used by all subsequent evaluation procedures.




In [ ]:
# ============================================================
# Step 4: Prepare Evaluation Dataset
# ============================================================

import pandas as pd
import numpy as np

print("Preparing evaluation dataset...\n")

# ------------------------------------------------------------
# Create Working Evaluation Dataset
# ------------------------------------------------------------

evaluation_df = baseline_predictions_df.copy()

# ------------------------------------------------------------
# Normalize Text Fields
# ------------------------------------------------------------

text_columns = [
    "question",
    "ground_truth",
    "prediction",
    "ground_truth_choice",
    "predicted_choice",
]

for column in text_columns:

    if column in evaluation_df.columns:

        evaluation_df[column] = (
            evaluation_df[column]
            .astype(str)
            .fillna("")
            .str.strip()
        )

# ------------------------------------------------------------
# Normalize Answer Mode
# ------------------------------------------------------------

evaluation_df["answer_mode"] = (
    evaluation_df["answer_mode"]
    .astype(str)
    .fillna("open_text")
    .str.strip()
)

# ------------------------------------------------------------
# Normalize Choice Correctness
# ------------------------------------------------------------

evaluation_df["choice_correct"] = (
    evaluation_df["choice_correct"]
    .astype(str)
    .str.lower()
    .map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        }
    )
)

evaluation_df["choice_correct"] = (
    evaluation_df["choice_correct"]
    .fillna(False)
)

# ------------------------------------------------------------
# Validate Choice Values
# ------------------------------------------------------------

valid_choice_values = {
    "0",
    "1",
    "2",
    "3",
    "4",
}

evaluation_df["valid_ground_truth_choice"] = (
    evaluation_df["ground_truth_choice"]
    .isin(valid_choice_values)
)

evaluation_df["valid_predicted_choice"] = (
    evaluation_df["predicted_choice"]
    .isin(valid_choice_values)
)

evaluation_df["is_multiple_choice"] = (
    evaluation_df["answer_mode"]
    == "multiple_choice"
)

evaluation_df["valid_multiple_choice_record"] = (
    evaluation_df["is_multiple_choice"]
    & evaluation_df["valid_ground_truth_choice"]
    & evaluation_df["valid_predicted_choice"]
)

# ------------------------------------------------------------
# Answer Length Statistics
# ------------------------------------------------------------

evaluation_df["ground_truth_length"] = (
    evaluation_df["ground_truth"]
    .astype(str)
    .str.len()
)

evaluation_df["prediction_length"] = (
    evaluation_df["prediction"]
    .astype(str)
    .str.len()
)

# ------------------------------------------------------------
# Display Dataset Summary
# ------------------------------------------------------------

print("Evaluation dataset prepared.")
print("-" * 60)
print(
    f"Total evaluation records           : "
    f"{len(evaluation_df):,}"
)
print(
    f"Multiple-choice records            : "
    f"{evaluation_df['is_multiple_choice'].sum():,}"
)
print(
    f"Valid multiple-choice predictions  : "
    f"{evaluation_df['valid_multiple_choice_record'].sum():,}"
)
print(
    f"Invalid multiple-choice predictions: "
    f"{(evaluation_df['is_multiple_choice'] & ~evaluation_df['valid_predicted_choice']).sum():,}"
)

print("\nPrepared Dataset Columns")
print("-" * 60)
print(list(evaluation_df.columns))

display(evaluation_df.head())



### 🔷 Step 5 — Verify Prediction Quality

* Validate prediction completeness and identify missing, empty, or invalid responses.
* Normalize ground-truth answers and model predictions for comparison.
* Detect exact-match and partial-match answer agreements.
* Measure prediction validity, coverage, and match rates.
* Generate prediction verification summaries and quality statistics.
* Prepare validated prediction results for subsequent evaluation metric computation.


In [ ]:
# ============================================================
# Step 5: Verify Prediction Quality
# ============================================================

import pandas as pd
import re

print("Verifying prediction quality...\n")

verification_df = evaluation_df.copy()

# ------------------------------------------------------------
# Text Normalization Helper
# ------------------------------------------------------------

def normalize_text(value):

    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

# ------------------------------------------------------------
# Normalize Ground Truth and Prediction Text
# ------------------------------------------------------------

verification_df["ground_truth_normalized"] = (
    verification_df["ground_truth"]
    .apply(normalize_text)
)

verification_df["prediction_normalized"] = (
    verification_df["prediction"]
    .apply(normalize_text)
)

# ------------------------------------------------------------
# Secondary Text-Based Metrics
# ------------------------------------------------------------

verification_df["exact_match"] = (
    verification_df["ground_truth_normalized"]
    == verification_df["prediction_normalized"]
)

verification_df["partial_match"] = verification_df.apply(
    lambda row: (
        row["ground_truth_normalized"] in row["prediction_normalized"]
        or row["prediction_normalized"] in row["ground_truth_normalized"]
    )
    if row["ground_truth_normalized"] and row["prediction_normalized"]
    else False,
    axis=1,
)

# ------------------------------------------------------------
# Primary Multiple-Choice Metric
# ------------------------------------------------------------

verification_df["choice_match"] = (
    verification_df["is_multiple_choice"]
    & verification_df["valid_predicted_choice"]
    & verification_df["choice_correct"]
)

# ------------------------------------------------------------
# Build Verification Summary
# ------------------------------------------------------------

verification_summary = {
    "total_predictions": len(verification_df),
    "multiple_choice_predictions": int(
        verification_df["is_multiple_choice"].sum()
    ),
    "valid_choice_predictions": int(
        verification_df["valid_multiple_choice_record"].sum()
    ),
    "invalid_choice_predictions": int(
        (
            verification_df["is_multiple_choice"]
            & ~verification_df["valid_predicted_choice"]
        ).sum()
    ),
    "correct_choice_predictions": int(
        verification_df["choice_match"].sum()
    ),
    "exact_matches": int(
        verification_df["exact_match"].sum()
    ),
    "partial_matches": int(
        verification_df["partial_match"].sum()
    ),
}

verification_summary_df = pd.DataFrame(
    verification_summary.items(),
    columns=[
        "metric",
        "value",
    ],
)

# ------------------------------------------------------------
# Display Verification Summary
# ------------------------------------------------------------

print("Prediction quality verification complete.")
print("-" * 60)

display(verification_summary_df)

print("\nSample Verification Records")
print("-" * 60)

display(
    verification_df[
        [
            "video",
            "question",
            "ground_truth_choice",
            "predicted_choice",
            "choice_correct",
            "exact_match",
            "partial_match",
        ]
    ].head(10)
)



### 🔷 Step 6 — Compute Evaluation Metrics

* Calculate overall evaluation metrics from validated prediction results.
* Measure exact-match accuracy, partial-match accuracy, and prediction coverage.
* Generate performance summaries by reasoning category.
* Generate performance summaries by NExT-QA question type.
* Analyze ground-truth and prediction answer lengths.
* Produce evaluation metrics used for comparative VideoQA analysis.

**Note:**
Exact-match and substring-based metrics provide a baseline evaluation framework but may underestimate VideoQA answer quality due to paraphrasing and semantic equivalence. Future work may incorporate semantic similarity metrics to better evaluate semantically equivalent answers expressed using different wording.



In [ ]:
# ============================================================
# Step 6: Compute Evaluation Metrics
# ============================================================

import pandas as pd
import numpy as np

print("Computing evaluation metrics...\n")

total_predictions = len(verification_df)

multiple_choice_df = verification_df[
    verification_df["is_multiple_choice"]
].copy()

valid_choice_df = multiple_choice_df[
    multiple_choice_df["valid_predicted_choice"]
].copy()

metric_records = []

def add_metric(metric, value):
    metric_records.append(
        {
            "metric": metric,
            "value": value,
        }
    )

# ------------------------------------------------------------
# Primary multiple-choice metrics
# ------------------------------------------------------------

if len(multiple_choice_df) > 0:
    choice_accuracy = (
        multiple_choice_df["choice_correct"].sum()
        / len(multiple_choice_df)
    )

    valid_choice_accuracy = (
        valid_choice_df["choice_correct"].sum()
        / len(valid_choice_df)
        if len(valid_choice_df) > 0
        else 0.0
    )

    add_metric("primary_metric", "choice_accuracy")
    add_metric("answer_mode", "multiple_choice")
    add_metric("total_predictions", total_predictions)
    add_metric("multiple_choice_predictions", len(multiple_choice_df))
    add_metric("valid_choice_predictions", len(valid_choice_df))
    add_metric(
        "invalid_choice_predictions",
        len(multiple_choice_df) - len(valid_choice_df),
    )
    add_metric(
        "correct_choice_predictions",
        int(multiple_choice_df["choice_correct"].sum()),
    )
    add_metric("choice_accuracy", choice_accuracy)
    add_metric("valid_choice_accuracy", valid_choice_accuracy)

else:
    exact_accuracy = verification_df["exact_match"].sum() / total_predictions

    add_metric("primary_metric", "exact_match_accuracy")
    add_metric("answer_mode", "open_text")
    add_metric("total_predictions", total_predictions)
    add_metric("exact_match_accuracy", exact_accuracy)

# ------------------------------------------------------------
# Secondary text metrics
# ------------------------------------------------------------

exact_match_accuracy = (
    verification_df["exact_match"].sum() / total_predictions
    if total_predictions > 0
    else 0.0
)

partial_match_accuracy = (
    verification_df["partial_match"].sum() / total_predictions
    if total_predictions > 0
    else 0.0
)

add_metric("exact_matches", int(verification_df["exact_match"].sum()))
add_metric("exact_match_accuracy", exact_match_accuracy)
add_metric("partial_matches", int(verification_df["partial_match"].sum()))
add_metric("partial_match_accuracy", partial_match_accuracy)

evaluation_metrics_df = pd.DataFrame(metric_records)

# ------------------------------------------------------------
# Category-level metrics
# ------------------------------------------------------------

category_metrics_records = []

if "type" in verification_df.columns:
    category_column = "type"
elif "question_type" in verification_df.columns:
    category_column = "question_type"
else:
    category_column = None

if category_column is not None:
    for category, category_df in verification_df.groupby(category_column):
        record = {
            category_column: category,
            "count": len(category_df),
            "exact_match_accuracy": category_df["exact_match"].mean(),
            "partial_match_accuracy": category_df["partial_match"].mean(),
        }

        if category_df["is_multiple_choice"].any():
            mc_category_df = category_df[category_df["is_multiple_choice"]]
            record["choice_accuracy"] = mc_category_df["choice_correct"].mean()

        category_metrics_records.append(record)

category_metrics_df = pd.DataFrame(category_metrics_records)

print("Evaluation metrics computed.")
display(evaluation_metrics_df)

if not category_metrics_df.empty:
    print("\nCategory metrics:")
    display(category_metrics_df)



### 🔷 Step 7 — Generate Runtime Analysis

* Analyze VideoQA experiment execution performance using recorded runtime statistics.
* Summarize total runtime and average inference time per sample.
* Estimate processing requirements for larger evaluation workloads.
* Calculate projected runtimes for validation-split and full-dataset execution.
* Generate runtime analysis summaries for evaluation reporting and experiment comparison.



In [ ]:
# ============================================================
# Step 7: Generate Runtime Analysis
# ============================================================

print("Generating runtime analysis...\n")

# ------------------------------------------------------------
# Helper Function for Summary Metric Lookup
# ------------------------------------------------------------

def get_metric(metric_name, default=None):
    metric_rows = baseline_summary_df[
        baseline_summary_df["metric"] == metric_name
    ]

    if metric_rows.empty:
        return default

    return metric_rows["value"].iloc[0]

# ------------------------------------------------------------
# Extract Runtime Metrics
# ------------------------------------------------------------

runtime_analysis = {
    "elapsed_time_seconds":
        get_metric("elapsed_time_seconds"),

    "average_time_per_sample_seconds":
        get_metric("average_time_per_sample_seconds"),

    "projected_validation_runtime_minutes":
        get_metric("projected_validation_runtime_minutes"),

    "projected_full_dataset_runtime_hours":
        get_metric("projected_full_dataset_runtime_hours"),

    "total_predictions":
        get_metric("total_predictions"),

    "valid_predictions":
        get_metric("valid_predictions"),
}

runtime_analysis_df = pd.DataFrame(
    list(runtime_analysis.items()),
    columns=["metric", "value"]
)

# ------------------------------------------------------------
# Add Human-Readable Runtime Values
# ------------------------------------------------------------

elapsed_seconds = float(
    runtime_analysis["elapsed_time_seconds"]
)

average_seconds = float(
    runtime_analysis["average_time_per_sample_seconds"]
)

validation_minutes = float(
    runtime_analysis["projected_validation_runtime_minutes"]
)

full_dataset_hours = float(
    runtime_analysis["projected_full_dataset_runtime_hours"]
)

runtime_summary_df = pd.DataFrame(
    [
        {
            "runtime_metric": "Elapsed runtime",
            "value": elapsed_seconds,
            "unit": "seconds",
        },
        {
            "runtime_metric": "Average runtime per sample",
            "value": average_seconds,
            "unit": "seconds/sample",
        },
        {
            "runtime_metric": "Projected validation split runtime",
            "value": validation_minutes,
            "unit": "minutes",
        },
        {
            "runtime_metric": "Projected full dataset runtime",
            "value": full_dataset_hours,
            "unit": "hours",
        },
    ]
)

# ------------------------------------------------------------
# Display Runtime Analysis
# ------------------------------------------------------------

print("Runtime Analysis Summary")
print("-" * 60)

display(runtime_summary_df)

print("\nRuntime Interpretation")
print("-" * 60)
print(
    f"The baseline run processed "
    f"{int(runtime_analysis['valid_predictions']):,} valid samples "
    f"in {elapsed_seconds:.2f} seconds."
)

print(
    f"Average runtime was "
    f"{average_seconds:.2f} seconds per sample."
)

print(
    f"Projected runtime for the full validation split is "
    f"{validation_minutes:.2f} minutes."
)

print(
    f"Projected runtime for the full dataset is "
    f"{full_dataset_hours:.2f} hours."
)



### 🔷 Step 8 — Create Visualizations

* Generate visualizations summarizing VideoQA evaluation results.
* Visualize overall evaluation metrics and prediction-quality statistics.
* Analyze performance across NExT-QA reasoning categories.
* Compare ground-truth and predicted answer characteristics.
* Visualize experiment runtime and projected execution requirements.
* Create publication-ready figures suitable for reports, presentations, and experiment documentation.
* Display generated visualizations within the notebook and save figure files for export.

In [ ]:
# ============================================================
# Step 8: Create Visualizations
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

print("Creating evaluation visualizations...\n")

# ------------------------------------------------------------
# Figure Output Directory
# ------------------------------------------------------------

figure_output_dir = Path(
    "outputs/evaluation/figures"
)

figure_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

generated_figures = []

def save_current_figure(filename):

    output_path = figure_output_dir / filename

    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

    generated_figures.append(
        {
            "figure": filename,
            "path": str(output_path),
        }
    )

# ------------------------------------------------------------
# Primary Metric Summary
# ------------------------------------------------------------

primary_metrics = []

if "multiple_choice" in verification_df["answer_mode"].unique():

    primary_metrics.append(
        {
            "metric": "Choice Accuracy",
            "value": verification_df.loc[
                verification_df["is_multiple_choice"],
                "choice_correct",
            ].mean(),
        }
    )

primary_metrics.extend(
    [
        {
            "metric": "Exact Match",
            "value": verification_df["exact_match"].mean(),
        },
        {
            "metric": "Partial Match",
            "value": verification_df["partial_match"].mean(),
        },
    ]
)

primary_metric_plot_df = pd.DataFrame(primary_metrics)

plt.figure(figsize=(8, 5))
plt.bar(
    primary_metric_plot_df["metric"],
    primary_metric_plot_df["value"],
)
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Evaluation Metric Summary")
plt.xticks(rotation=20)
save_current_figure("evaluation_metrics_summary.png")

# ------------------------------------------------------------
# Multiple-Choice Prediction Distribution
# ------------------------------------------------------------

if "multiple_choice" in verification_df["answer_mode"].unique():

    choice_counts = (
        verification_df.loc[
            verification_df["is_multiple_choice"],
            "predicted_choice",
        ]
        .value_counts()
        .sort_index()
    )

    plt.figure(figsize=(8, 5))
    plt.bar(
        choice_counts.index.astype(str),
        choice_counts.values,
    )
    plt.xlabel("Predicted Choice")
    plt.ylabel("Count")
    plt.title("Predicted Multiple-Choice Answer Distribution")
    save_current_figure("predicted_choice_distribution.png")

# ------------------------------------------------------------
# Correct vs Incorrect Choice Counts
# ------------------------------------------------------------

if "multiple_choice" in verification_df["answer_mode"].unique():

    correctness_counts = (
        verification_df.loc[
            verification_df["is_multiple_choice"],
            "choice_correct",
        ]
        .value_counts()
        .reindex([True, False], fill_value=0)
    )

    correctness_labels = [
        "Correct",
        "Incorrect",
    ]

    plt.figure(figsize=(7, 5))
    plt.bar(
        correctness_labels,
        correctness_counts.values,
    )
    plt.ylabel("Count")
    plt.title("Multiple-Choice Correctness Summary")
    save_current_figure("choice_correctness_summary.png")

# ------------------------------------------------------------
# Category Performance
# ------------------------------------------------------------

if "category_metrics_df" in globals() and not category_metrics_df.empty:

    category_label = category_metrics_df.columns[0]

    metric_to_plot = (
        "choice_accuracy"
        if "choice_accuracy" in category_metrics_df.columns
        else "exact_match_accuracy"
    )

    plot_df = category_metrics_df.sort_values(
        metric_to_plot,
        ascending=False,
    )

    plt.figure(figsize=(10, 5))
    plt.bar(
        plot_df[category_label].astype(str),
        plot_df[metric_to_plot],
    )
    plt.ylim(0, 1)
    plt.xlabel(category_label)
    plt.ylabel("Accuracy")
    plt.title(
        f"{metric_to_plot.replace('_', ' ').title()} "
        f"by {category_label}"
    )
    plt.xticks(rotation=30)
    save_current_figure("category_performance.png")

# ------------------------------------------------------------
# Answer Length Comparison
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))
plt.hist(
    verification_df["ground_truth_length"],
    alpha=0.7,
    label="Ground Truth",
)
plt.hist(
    verification_df["prediction_length"],
    alpha=0.7,
    label="Prediction",
)
plt.xlabel("Text Length")
plt.ylabel("Count")
plt.title("Ground Truth vs Prediction Text Length")
plt.legend()
save_current_figure("text_length_comparison.png")

# ------------------------------------------------------------
# Runtime Summary
# ------------------------------------------------------------

if "runtime_analysis_df" in globals() and not runtime_analysis_df.empty:

    runtime_plot_df = runtime_analysis_df.copy()

    if "runtime_metric" in runtime_plot_df.columns:
        runtime_label_column = "runtime_metric"
    elif "metric" in runtime_plot_df.columns:
        runtime_label_column = "metric"
    else:
        runtime_label_column = runtime_plot_df.columns[0]

    if "value" in runtime_plot_df.columns:
        runtime_value_column = "value"
    else:
        runtime_value_column = runtime_plot_df.columns[1]

    runtime_plot_df[runtime_value_column] = pd.to_numeric(
        runtime_plot_df[runtime_value_column],
        errors="coerce",
    )

    runtime_plot_df = runtime_plot_df.dropna(
        subset=[runtime_value_column]
    )

    if not runtime_plot_df.empty:

        plt.figure(figsize=(10, 5))
        plt.bar(
            runtime_plot_df[runtime_label_column].astype(str),
            runtime_plot_df[runtime_value_column],
        )
        plt.ylabel("Value")
        plt.title("Runtime Analysis Summary")
        plt.xticks(rotation=30, ha="right")
        save_current_figure("runtime_analysis_summary.png")

# ------------------------------------------------------------
# Generated Figure Summary
# ------------------------------------------------------------

generated_figures_df = pd.DataFrame(
    generated_figures
)

print("Generated figures:")
display(generated_figures_df)



### 🔷 Step 9 — Save Evaluation Results

* Save the unified evaluation dataset to the project output directory.
* Save prediction verification summaries and evaluation metric tables.
* Save reasoning-category, question-type, answer-length, and runtime analysis results.
* Save generated figure metadata for tracking visualization outputs.
* Preserve evaluation artifacts for later reporting, comparison, and project documentation.



In [ ]:
# ============================================================
# Step 9: Save Evaluation Results
# ============================================================

import os

print("Saving evaluation results...\n")

# ------------------------------------------------------------
# Create Output Directory
# ------------------------------------------------------------

evaluation_report_dir = (
    "outputs/evaluation/reports"
)

os.makedirs(
    evaluation_report_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# Define Output Files
# ------------------------------------------------------------

evaluation_dataset_file = os.path.join(
    evaluation_report_dir,
    "evaluation_dataset.csv"
)

verification_results_file = os.path.join(
    evaluation_report_dir,
    "verification_results.csv"
)

verification_summary_file = os.path.join(
    evaluation_report_dir,
    "verification_summary.csv"
)

evaluation_metrics_file = os.path.join(
    evaluation_report_dir,
    "evaluation_metrics.csv"
)

runtime_analysis_file = os.path.join(
    evaluation_report_dir,
    "runtime_analysis.csv"
)

generated_figures_file = os.path.join(
    evaluation_report_dir,
    "generated_figures.csv"
)

category_metrics_file = os.path.join(
    evaluation_report_dir,
    "category_metrics.csv"
)

# ------------------------------------------------------------
# Save Core Evaluation Results
# ------------------------------------------------------------

evaluation_df.to_csv(
    evaluation_dataset_file,
    index=False
)

verification_df.to_csv(
    verification_results_file,
    index=False
)

verification_summary_df.to_csv(
    verification_summary_file,
    index=False
)

evaluation_metrics_df.to_csv(
    evaluation_metrics_file,
    index=False
)

runtime_analysis_df.to_csv(
    runtime_analysis_file,
    index=False
)

generated_figures_df.to_csv(
    generated_figures_file,
    index=False
)

# ------------------------------------------------------------
# Save Optional Category Metrics
# ------------------------------------------------------------

saved_files = [
    evaluation_dataset_file,
    verification_results_file,
    verification_summary_file,
    evaluation_metrics_file,
    runtime_analysis_file,
    generated_figures_file,
]

if (
    "category_metrics_df" in globals()
    and not category_metrics_df.empty
):
    category_metrics_df.to_csv(
        category_metrics_file,
        index=False
    )

    saved_files.append(
        category_metrics_file
    )

# ------------------------------------------------------------
# Display Saved Files
# ------------------------------------------------------------

print("Saved Evaluation Files")
print("-" * 60)

for file_path in saved_files:

    file_size_kb = (
        os.path.getsize(file_path)
        / 1024
    )

    print(
        f"{os.path.basename(file_path):<35}"
        f"{file_size_kb:8.1f} KB"
    )

print("\nEvaluation results saved successfully.")



### 🔷 Step 10 — Display Evaluation Results

* Display overall evaluation metrics and prediction verification summaries.
* Present reasoning-category, question-type, answer-length, and runtime analyses.
* Display generated visualization summaries and saved figure information.
* Review key experiment findings and performance statistics.
* Summarize prediction quality, evaluation accuracy, and runtime characteristics.
* Confirm that evaluation outputs, reports, and visualization artifacts were successfully generated.


In [ ]:
# ============================================================
# Step 10: Display Evaluation Results
# ============================================================

print("Displaying evaluation results...\n")

# ------------------------------------------------------------
# Evaluation Metrics
# ------------------------------------------------------------

print("Evaluation Metrics")
print("-" * 60)

display(
    evaluation_metrics_df
)

# ------------------------------------------------------------
# Category Metrics (Optional)
# ------------------------------------------------------------

if (
    "category_metrics_df" in globals()
    and not category_metrics_df.empty
):

    print("\nCategory Metrics")
    print("-" * 60)

    display(
        category_metrics_df
    )

# ------------------------------------------------------------
# Verification Summary
# ------------------------------------------------------------

print("\nPrediction Verification Summary")
print("-" * 60)

display(
    verification_summary_df
)

# ------------------------------------------------------------
# Runtime Analysis
# ------------------------------------------------------------

if (
    "runtime_analysis_df" in globals()
    and not runtime_analysis_df.empty
):

    print("\nRuntime Analysis")
    print("-" * 60)

    display(
        runtime_analysis_df
    )

# ------------------------------------------------------------
# Sample Predictions
# ------------------------------------------------------------

display_columns = [
    "video",
    "question",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "ground_truth",
    "prediction",
    "exact_match",
    "partial_match",
]

available_display_columns = [
    column
    for column in display_columns
    if column in verification_df.columns
]

print("\nSample Evaluated Predictions")
print("-" * 60)

sample_count = min(
    10,
    len(verification_df),
)

display(
    verification_df[
        available_display_columns
    ]
    .sample(
        n=sample_count,
        random_state=42,
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Generated Figures
# ------------------------------------------------------------

if (
    "generated_figures_df" in globals()
    and not generated_figures_df.empty
):

    print("\nGenerated Figures")
    print("-" * 60)

    display(
        generated_figures_df
    )

# ------------------------------------------------------------
# Evaluation Summary
# ------------------------------------------------------------

print("\nEvaluation Summary")
print("-" * 60)

choice_accuracy_row = evaluation_metrics_df[
    evaluation_metrics_df["metric"]
    == "choice_accuracy"
]

if not choice_accuracy_row.empty:

    choice_accuracy = float(
        choice_accuracy_row["value"].iloc[0]
    )

    print(
        f"Multiple-Choice Accuracy : "
        f"{choice_accuracy:.2%}"
    )

correct_predictions_row = evaluation_metrics_df[
    evaluation_metrics_df["metric"]
    == "correct_choice_predictions"
]

if not correct_predictions_row.empty:

    print(
        f"Correct Predictions      : "
        f"{int(correct_predictions_row['value'].iloc[0])}"
    )

total_predictions_row = evaluation_metrics_df[
    evaluation_metrics_df["metric"]
    == "total_predictions"
]

if not total_predictions_row.empty:

    print(
        f"Total Predictions        : "
        f"{int(total_predictions_row['value'].iloc[0])}"
    )

print("\nNotebook 06 evaluation complete.")

